In [64]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve

In [65]:
train_scores = pd.read_csv('../Model_scores/scores_train.csv')
test_scores = pd.read_csv('../Model_scores/scores_test.csv')
oot_scores = pd.read_csv('../Model_scores/scores_oot.csv')

categorical_features = pd.read_csv('../Model_scores/categorical_features.csv')

In [66]:
train_scores['bins'] = pd.qcut(train_scores['y_pred'], q=10, labels=False, duplicates='drop')
score_agg = (
    train_scores.groupby('bins')
    .agg(
        y_true=('y_true', 'mean'),
        y_pred_mean=('y_pred', 'mean'),
        lower_bound=('y_pred', 'min'),
        upper_bound=('y_pred', 'max')
    )
    .reset_index()
)
score_agg['diff'] = score_agg['y_pred_mean'] - score_agg['y_true']
score_agg

,bins,y_true,y_pred_mean,lower_bound,upper_bound,diff
0,0,0.000126,0.000775,0.000143,0.001109,0.000649
1,1,0.000505,0.001446,0.001109,0.001811,0.000941
2,2,0.000883,0.002246,0.001811,0.002733,0.001362
3,3,0.001578,0.003338,0.002733,0.004009,0.001761
4,4,0.002777,0.004820,0.004009,0.005710,0.002043
5,5,0.004649,0.006764,0.005710,0.007903,0.002115
6,6,0.007110,0.009297,0.007903,0.010888,0.002187
7,7,0.010455,0.013170,0.010888,0.015973,0.002715
8,8,0.020573,0.020961,0.015973,0.028185,0.000388
9,9,0.091230,0.077015,0.028185,0.933584,-0.014215


In [57]:
def assign_bins(score):
    for _, row in score_agg.iterrows():
        if score <= row['upper_bound']:
            return int(row['bins'])
    return np.nan

In [58]:
test_scores['bins'] = test_scores['y_pred'].apply(assign_bins)

In [61]:
test_scores['cablirated_pd'] = test_scores['bins'].apply(lambda x: score_agg.loc[score_agg['bins'] == x, 'y_true'].values[0] if pd.notnull(x) else np.nan)

In [63]:
test_scores.groupby('bins').agg(
    y_true=('y_true', 'mean'),
    y_pred_mean=('y_pred', 'mean'),
    calibrated_pd=('cablirated_pd', 'mean')
).reset_index()

,bins,y_true,y_pred_mean,calibrated_pd
0,0,0.000769,0.000767,0.000126
1,1,0.001726,0.001446,0.000505
2,2,0.002520,0.002249,0.000883
3,3,0.005062,0.003335,0.001578
4,4,0.007271,0.004825,0.002777
5,5,0.010524,0.006750,0.004649
6,6,0.012452,0.009283,0.007110
7,7,0.020138,0.013134,0.010455
8,8,0.031238,0.021008,0.020573
9,9,0.090795,0.075215,0.091230
